# 32. Introductory model inspection
Inspection asks how a fitted model uses inputs. Permutation importance measures how an evaluation score changes when a feature’s values are shuffled. Partial dependence averages model predictions as a selected feature is varied. Neither method proves causality, and correlated features can make interpretation difficult.

Use an appropriate held-out development partition for inspection when results guide improvements. Repeat permutations to inspect variation. Correlated substitutes can make each feature appear individually less important. Partial dependence can evaluate unrealistic feature combinations when inputs are strongly related. Report the model, scoring rule, data, and these limitations.

In [ ]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
X, y = make_regression(n_samples=160, n_features=4, noise=12, random_state=42)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance, partial_dependence
Xt, Xv, yt, yv = train_test_split(X_dev, y_dev, random_state=7) # Development-only inspection split.
model = RandomForestRegressor(n_estimators=30, random_state=42).fit(Xt, yt) # Fit a small inspectable model.
result = permutation_importance(model, Xv, yv, n_repeats=3, random_state=42, scoring="neg_mean_absolute_error") # Repeat feature shuffles.
print("Mean importances:", result.importances_mean.round(2)) # Positive means shuffling worsened score on average.
pd_result = partial_dependence(model, Xv, features=[0], grid_resolution=5) # Average predictions over a feature grid.
print("Partial dependence shape:", pd_result["average"].shape) # Inspect response dimensions.
assert len(result.importances_mean) == X.shape[1] # One importance per input feature.

## Permutation importance, PDP, and ICE in pictures
The existing example already computes permutation importance; its label “Mean importances” means mean score changes over repeated permutations, not impurity importance. This lab visualizes that variation and adds ICE. A PDP averages model response across examples, which can hide opposing individual patterns. ICE draws a response for each selected example; kind="both" overlays those curves and their mean. Both methods can create implausible feature combinations when inputs depend on each other. Neither represents a causal intervention.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_friedman1
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
X, y = make_friedman1(n_samples=180, n_features=5, noise=1, random_state=42)
Xt, Xv, yt, yv = train_test_split(X, y, random_state=42) # Development inspection partition.
model = RandomForestRegressor(n_estimators=30, min_samples_leaf=3, random_state=42).fit(Xt, yt)
importance = permutation_importance(model, Xv, yv, n_repeats=5, random_state=42, scoring="neg_mean_absolute_error") # Score loss after shuffling.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(range(5), importance.importances_mean, yerr=importance.importances_std) # Descriptive repeat variation.
axes[0].set(xlabel="Feature index", ylabel="MAE score decrease", title="Permutation importance")
PartialDependenceDisplay.from_estimator(model, Xv, [0], kind="both", method="brute", subsample=20, random_state=42, grid_resolution=12, ax=axes[1]) # Individual curves and their average.
axes[1].set_title("PDP + ICE for feature 0")
fig.tight_layout(); fig.savefig("inspection-depth.png"); plt.close(fig)
print("Permutation means:", importance.importances_mean.round(3).tolist())
assert importance.importances.shape == (5, 5) # Five features by five permutations.

### Reading the result
Positive permutation importance means shuffling reduced the chosen score on average. Error bars show repeat variability, not a confidence interval. ICE curves describe model response for individual feature contexts; they need not be parallel. Plotting 20 rows is a readability choice, not a representative subgroup analysis.

### Transfer exercise
Compare PDP and ICE for a feature interacting with another feature. Describe what averaging hides and check whether the feature combinations are plausible.

## Your turn
Write two sentences describing a feature’s model importance and one sentence explaining why it is not a causal conclusion.

Try the task before opening the solution on the course website.